# A09 · Serving, Inference Optimization & Cost

**Track:** Advanced (lesson 9 of 10) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free (it is mostly simulation), or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** A01 (LLM APIs), A07 (evals) · **You will build:** a latency model, a KV-cache memory calculator, a quantization-error demo, a prompt-caching simulator and a model cascade that you evaluate on quality *and* cost

---
## 🎯 Why this matters
A prototype that costs $0.04 per request and takes 9 seconds is a demo. The same product at 50,000 requests a day is a $2,000/day line item and a UX problem. Inference engineering is
the set of levers that move cost and latency by 2–10× **without** (visibly) moving quality, and the discipline is to *measure the quality side of every trade*.

**Real scenario.** *Tāmaki Insurance* runs a claims-triage assistant: each request carries a 6,000-token policy-wording prefix plus a 300-token claim note and returns ~250 tokens. It
handles 40,000 requests a day, most of them routine. Finance asks: "can we halve the bill without the adjusters noticing?"

## 🧭 Learning objectives
By the end you can:
1. Decompose request latency into queueing, prefill (TTFT) and decode (tokens/s), and say which one a given optimisation attacks.
2. Compute KV-cache memory from a model configuration and explain why it, not weights, limits concurrency.
3. Measure the error introduced by weight quantization and explain why outliers and group size matter.
4. Simulate prompt-caching economics and find the request rate at which it pays off; list what breaks a cache.
5. Build a router/cascade and quantify cost saved versus quality lost on a labelled workload.

> **Honesty note.** Hardware/latency constants below are *illustrative parameters* chosen to show shapes, not benchmarks of any product. Money figures come from `agentkit.trace.PRICES`;
> caching and batch multipliers are parameters with their documented structure noted: confirm current numbers on the pricing page.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: the anatomy of one request
An autoregressive model generates one token at a time. A request therefore has two very different phases:

```
 request ─▶ [ queue ] ─▶ [ PREFILL: read all n_in prompt tokens in parallel ] ─▶ first token ─▶ [ DECODE: 1 token per step, n_out-1 steps ] ─▶ done
                          └────────────── TTFT (time to first token) ─────────┘   └──── (n_out-1) / tokens-per-second ───────────┘
```

* **Prefill** is compute-bound and parallel: long prompts cost milliseconds per thousand tokens. It sets **TTFT**, the number users feel when streaming.
* **Decode** is memory-bandwidth-bound and sequential: every new token must stream the model weights (and the KV cache) through the chip. It sets **tokens per second**.
* **End-to-end** `T = queue + n_in / prefill_rate + (n_out − 1) / decode_tps`.

Different products care about different terms: a chat UI is dominated by TTFT, a report generator by decode time, a batch classifier by neither (only throughput per dollar).

In [ ]:
import math, random, heapq
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from agentkit.trace import PRICES
from agentkit import get_llm, say

# Illustrative constants (NOT benchmarks): a mid-size hosted model
OVERHEAD_S, PREFILL_TPS, DECODE_TPS = 0.20, 8000.0, 60.0

def latency(n_in, n_out, queue=0.0):
    ttft = queue + OVERHEAD_S + n_in / PREFILL_TPS
    return {"ttft": ttft, "decode": (n_out - 1) / DECODE_TPS, "total": ttft + (n_out - 1) / DECODE_TPS}

WORKLOADS = {"chat turn": (500, 150), "RAG answer (6k ctx)": (6300, 250), "long summary": (20000, 800), "classify": (400, 5)}
fig, ax = plt.subplots(figsize=(7, 3))
for i, (name, (n_in, n_out)) in enumerate(WORKLOADS.items()):
    L = latency(n_in, n_out)
    ax.barh(i, L["ttft"], color="#4c78a8", label="TTFT" if i == 0 else None); ax.barh(i, L["decode"], left=L["ttft"], color="#f58518", label="decode" if i == 0 else None)
    print(f"{name:22s} TTFT {L['ttft']:.2f}s  decode {L['decode']:.2f}s  total {L['total']:.2f}s")
ax.set_yticks(range(len(WORKLOADS))); ax.set_yticklabels(WORKLOADS); ax.set_xlabel("seconds"); ax.legend(); plt.tight_layout(); plt.show()

assert latency(20000, 5)["ttft"] > latency(20000, 5)["decode"]          # long prompt, short answer: prefill dominates
assert latency(500, 800)["decode"] > 10 * latency(500, 800)["ttft"]     # short prompt, long answer: decode dominates

**Which lever hits which term?**

| Lever | Attacks | Notes |
|---|---|---|
| Shorter prompts, **prompt caching** | prefill / TTFT, and input $ | cached prefix is not re-computed |
| Streaming | *perceived* latency | total time is unchanged; the user sees TTFT |
| Fewer output tokens (tight schema, `max_tokens`) | decode | output tokens are slowest *and* priciest |
| Smaller / quantized model, **routing** | both, plus $ | quality risk: must be measured |
| Batching | throughput per GPU, not single-request latency | (next section) |

## 2 · Batching: why serving is a throughput game
Decode streams the whole weight set from memory for *every step*. If 32 requests share that step, each pays 1/32 of the weight traffic. Model a decode step as
`t(b) = t_weights + b · t_per_seq`; the first term is shared, the second is per sequence. Throughput `b / t(b)` rises with batch size, but each user's tokens/s falls.

In [ ]:
T_W, T_SEQ = 0.020, 0.0006        # seconds: shared weight read per step, per-sequence cost per step (illustrative)
bs = np.array([1, 2, 4, 8, 16, 32, 64, 128])
step = T_W + bs * T_SEQ
thr, per_user = bs / step, 1 / step
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(bs, thr, "o-"); ax[0].set_xscale("log", base=2); ax[0].set_title("server throughput (tok/s)"); ax[0].grid(alpha=.3)
ax[1].plot(bs, per_user, "o-", color="#f58518"); ax[1].set_xscale("log", base=2); ax[1].set_title("per-user speed (tok/s)"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"batch 1: {thr[0]:.0f} tok/s total, {per_user[0]:.0f}/user | batch 64: {thr[6]:.0f} tok/s total, {per_user[6]:.0f}/user")
assert thr[6] > 10 * thr[0] and per_user[6] < per_user[0]

**Static vs continuous batching.** Requests finish at different lengths. *Static* batching waits for the slowest request in the batch, so finished slots idle. *Continuous* (iteration-level)
batching refills a slot the moment its request ends. Simulate it with skewed output lengths (most answers short, a few long):

In [ ]:
rs = np.random.default_rng(0)
lengths = np.clip(rs.lognormal(mean=4.3, sigma=0.8, size=2000), 5, 1500).astype(int)   # median ~ 74 tokens, long tail
B = 32

def static_steps(lengths, B):
    return sum(lengths[i:i + B].max() for i in range(0, len(lengths), B))

def continuous_steps(lengths, B):
    q, active, steps = list(lengths), [], 0
    while q or active:
        while q and len(active) < B: active.append(q.pop())
        steps += 1
        active = [x - 1 for x in active if x - 1 > 0]
    return steps

s_static, s_cont = static_steps(lengths, B), continuous_steps(lengths, B)
ideal = lengths.sum() / B
print(f"decode steps  static: {s_static}  continuous: {s_cont}  ideal (no idle slots): {ideal:.0f}")
print(f"slot utilisation  static: {ideal / s_static:.0%}  continuous: {ideal / s_cont:.0%}  -> continuous is {s_static / s_cont:.2f}x faster")
assert s_cont < s_static and ideal / s_cont > 0.95

Production engines (for example vLLM, which popularised paged KV-cache memory management: Kwon et al., 2023, *Efficient Memory Management for Large Language Model Serving with PagedAttention*, arXiv:2309.06180)
implement continuous batching for you. If you call a hosted API you inherit it; if you self-host, this is the first thing to check.

For work that is **not interactive** (nightly classification, evals, embeddings) the major providers offer asynchronous *batch* endpoints at a discount in exchange for latency up to hours; check the current docs for the discount and limits. Route every offline job there.

## 3 · KV-cache memory: the real concurrency limit
During decode, each layer needs the keys and values of every previous token. Recomputing them is quadratic, so they are cached. Per token, per layer, we store one key and one value vector
for each **KV head**:

$$\text{KV bytes} = 2 \cdot L \cdot H_{kv} \cdot d_{head} \cdot s \cdot b \cdot p$$

with `L` layers, `H_kv` KV heads, `d_head` head dimension, `s` sequence length, `b` concurrent sequences and `p` bytes per element (2 for fp16/bf16, 1 for fp8/int8). The leading 2 is K and V.

**Worked example (an 8B-class configuration, hypothetical but typical):** `L=32, H_kv=8` (grouped-query attention), `d_head=128`, fp16.
Per token: 2·32·8·128·2 = **131,072 B = 128 KiB**. An 8,000-token context: ≈ **1 GiB per sequence**. On an 80 GB accelerator holding 16 GB of fp16 weights and reserving ~10% for activations,
about 56 GB remain, so roughly **53 concurrent 8k sequences (56e9 / 1.05e9), regardless of how fast the chip is**.

In [ ]:
def kv_bytes(layers, kv_heads, head_dim, seq, batch=1, bytes_per=2):
    return 2 * layers * kv_heads * head_dim * seq * batch * bytes_per

CFG = dict(layers=32, kv_heads=8, head_dim=128)
per_tok = kv_bytes(**CFG, seq=1)
GiB = 2 ** 30
print(f"per token: {per_tok:,} B = {per_tok / 1024:.0f} KiB | 8k context: {kv_bytes(**CFG, seq=8000) / GiB:.2f} GiB per sequence")

def max_concurrency(gpu_gb, weight_gb, seq, reserve=0.10, **cfg):
    free = gpu_gb * (1 - reserve) - weight_gb
    return int(free * 1e9 // kv_bytes(**cfg, seq=seq))

mha = dict(CFG, kv_heads=32)             # same model with full multi-head attention instead of GQA
for name, cfg, bp in [("GQA fp16 (H_kv=8)", CFG, 2), ("MHA fp16 (H_kv=32)", mha, 2), ("GQA fp8 KV", CFG, 1)]:
    n = max_concurrency(80, 16, 8000, **cfg, bytes_per=bp)
    print(f"{name:20s} -> {n:3d} concurrent 8k-token sequences on 80 GB (16 GB weights)")

assert per_tok == 131072
assert max_concurrency(80, 16, 8000, **mha) * 4 <= max_concurrency(80, 16, 8000, **CFG) + 4      # 4x the KV heads -> ~1/4 the concurrency
assert max_concurrency(80, 16, 8000, **CFG, bytes_per=1) >= 2 * max_concurrency(80, 16, 8000, **CFG) - 1

**Consequences you can now reason about:** doubling context halves concurrency; GQA (fewer KV heads) and KV quantization buy concurrency directly; a *shared prefix* (many requests with the
same 6k-token policy text) can share one copy of its KV blocks, which is also the idea behind prompt caching (section 5).

## 4 · Quantization: fewer bits per weight
Storing weights in 8 or 4 bits shrinks memory (and the weight traffic that bounds decode speed) by 2–4× versus fp16. The price is rounding error. A symmetric scheme maps real
values to integers with a scale: `q = round(w / s)`, `s = max|w| / (2^(bits-1) − 1)`. One **scale for the whole tensor** is simplest; one scale per **group** of values is more accurate
because a single outlier no longer inflates the step size for everyone.

Real LLM weights have rare large outliers, so we plant them. We measure the error of the weights *and* of the layer output `y = xW`, which is what the model actually experiences.

In [ ]:
rs = np.random.default_rng(1)
W = rs.normal(0, 0.02, size=(1024, 1024)).astype(np.float64)
mask = rs.random(W.shape) < 0.001
W[mask] *= 40                                   # ~0.1% outliers, 40x larger
x = rs.normal(0, 1, size=(256, 1024))

def quantize(W, bits, group=None):
    qmax = 2 ** (bits - 1) - 1
    Wg = W.reshape(-1, W.shape[1] if group is None else group)          # rows of length `group` (or whole rows)
    if group is None: Wg = W.reshape(1, -1)                              # per-tensor: a single scale
    s = np.abs(Wg).max(axis=1, keepdims=True) / qmax
    q = np.clip(np.round(Wg / s), -qmax - 1, qmax)
    return (q * s).reshape(W.shape)

def rel_err(a, b): return np.linalg.norm(a - b) / np.linalg.norm(b)
y_ref = x @ W
results = {}
for bits in (8, 4):
    for group in (None, 64):
        Wq = quantize(W, bits, group)
        results[(bits, group)] = (rel_err(Wq, W), rel_err(x @ Wq, y_ref))
        print(f"int{bits} {'per-tensor' if group is None else f'group={group:>3}':>12}  weight err {results[(bits, group)][0]:.4f}   output err {results[(bits, group)][1]:.4f}")

assert results[(8, 64)][1] < results[(8, None)][1] < results[(4, None)][1]
assert results[(4, 64)][1] < results[(4, None)][1] / 2, "grouping should at least halve int4 error with outliers present"
params = 8e9
print("\nweights memory for 8B params:", {f"{b}-bit": f"{params * b / 8 / 1e9:.0f} GB" for b in (16, 8, 4)})

**Takeaways from the numbers.** (1) 8-bit is nearly free; 4-bit per-tensor is visibly damaged by outliers. (2) Group-wise scales recover most of it at the cost of storing extra scales
(~0.25–0.5 bits/weight at group 64, depending on scale precision). (3) Weight error is a *proxy*: what you must measure is the **task metric** (your eval set, A07) before and after.
Methods such as GPTQ (Frantar et al., 2022, arXiv:2210.17323) and AWQ (Lin et al., 2023, arXiv:2306.00978) exist to choose the rounding more cleverly than `round()`.

> If you consume a hosted API you do not choose the precision; this matters when you self-host open-weight models, or when comparing "same model, cheaper host" offers: ask what precision they serve and re-run your eval.

## 5 · Prompt caching: the biggest lever for RAG and agents
Agents and RAG apps resend the *same* long prefix (system prompt, tool schemas, policy documents) on every call. With **prompt caching** the provider stores the KV state for that prefix; a later request that
begins with the identical prefix reads it at a steep discount and skips most of the prefill. The documented structure on Anthropic's API: you mark a cache breakpoint; the first request **writes** the cache at a
*premium* over normal input price; reads within the time-to-live are billed at a *fraction* of the input price; each read refreshes the TTL. (Check the docs for the current multipliers, minimum cacheable length and TTL options; we use parameters.)

The economics are therefore a bet on **reuse before expiry**. Simulate Tāmaki's workload across request rates.

In [ ]:
WRITE_MULT, READ_MULT, TTL_S = 1.25, 0.10, 300          # parameters: confirm on the pricing page
PREFIX, SUFFIX, OUT = 6000, 300, 250
MODEL = "claude-sonnet-5-5"; P_IN, P_OUT = PRICES[MODEL]

def simulate_cache(req_per_hour, hours=24, seed=0, ttl=TTL_S):
    r = np.random.default_rng(seed)
    n = r.poisson(req_per_hour * hours)
    t = np.sort(r.uniform(0, hours * 3600, n))
    plain = cached = 0.0; expires = -1.0; hits = 0
    for ti in t:
        base_out = OUT * P_OUT / 1e6
        plain += (PREFIX + SUFFIX) * P_IN / 1e6 + base_out
        if ti <= expires:   # hit: read the prefix cheaply, refresh TTL
            cached += PREFIX * P_IN * READ_MULT / 1e6 + SUFFIX * P_IN / 1e6 + base_out; hits += 1
        else:               # miss: write the prefix at a premium
            cached += PREFIX * P_IN * WRITE_MULT / 1e6 + SUFFIX * P_IN / 1e6 + base_out
        expires = ti + ttl
    return plain, cached, hits / max(1, n)

rates = [0.5, 1, 4, 12, 60, 600, 1700]
print(f"{'req/hour':>9} {'no cache $':>11} {'cached $':>9} {'saving':>7} {'hit rate':>9}")
sim = {}
for rate in rates:
    plain, cached, hr = simulate_cache(rate); sim[rate] = (plain, cached, hr)
    print(f"{rate:9.1f} {plain:11.2f} {cached:9.2f} {1 - cached / plain:7.0%} {hr:9.0%}")
assert sim[0.5][1] > sim[0.5][0], "at ~1 request per 2h the cache expires between uses and the write premium makes it MORE expensive"
assert sim[1700][1] < 0.45 * sim[1700][0] and sim[1700][2] > 0.99      # Tāmaki: 40,000/day ~ 1,700/hour

Two shapes to remember: below roughly one request per TTL window the cache **loses money** (you keep paying the write premium), and once requests arrive well inside the TTL (here, about one a minute) the saving saturates at
`(1 − read_mult)` of the *prefix* share of your bill. The output tokens are untouched, so the realised saving is smaller than the headline. Also note the latency benefit: cached tokens skip most prefill, so TTFT drops with them.

**What breaks a cache?** Matching is **by exact prefix**. Anything that changes an early token (a timestamp or user name at the top of the system prompt, reordered tool schemas, a different model)
invalidates everything after it. Let us watch it happen with a toy prefix matcher.

In [ ]:
def cached_prefix_len(prev: list[str], new: list[str]) -> int:
    n = 0
    for a, b in zip(prev, new):
        if a != b: break
        n += 1
    return n

policy = ("Section " + " clause").split() + [f"clause{i}" for i in range(600)]           # stand-in for 6k tokens of policy wording

def prompt_bad(now, claim):  return [f"Current-time:{now}"] + policy + ["CLAIM:"] + claim.split()     # volatile data FIRST
def prompt_good(now, claim): return policy + ["CLAIM:"] + claim.split() + [f"Current-time:{now}"]    # stable prefix first, volatile last

for name, mk in (("timestamp first", prompt_bad), ("timestamp last ", prompt_good)):
    a, b = mk("09:01", "water damage kitchen"), mk("09:02", "hail damage roof")
    k = cached_prefix_len(a, b)
    print(f"{name}: reusable prefix = {k} / {len(b)} tokens ({k / len(b):.0%})")
assert cached_prefix_len(prompt_bad("1", "x"), prompt_bad("2", "x")) == 0 and cached_prefix_len(prompt_good("1", "x"), prompt_good("2", "y")) > 600

## 6 · Model routing and cascades
Most traffic is easy. A cheap model answers easy queries as well as a strong one; only the hard tail needs the expensive model. Two designs:

* **Router**: classify the query *first* (difficulty/topic) and send it to one model. You pay one model plus the router.
* **Cascade**: always try the cheap model, check its answer (confidence, a verifier, a schema/consistency check), and **escalate** if unsure. You pay both models on escalated queries.

Either is a bet that your difficulty signal is *good*. So we build a synthetic workload with known ground truth and measure.

**Workload model.** 3,000 triage queries. Each has a latent difficulty `d ∈ [0,1]` (Beta(1.3,4): mostly easy). The chance a model answers correctly falls with `d` along a sigmoid; the strong model's curve sits to the right.
Each query has a fixed random draw `u`, so the same query is "correct" for model *m* iff `u < p_m(d)` (the strong model is right whenever the cheap one is, plus more). We then price tokens with `PRICES`.

In [ ]:
N = 3000
rs = np.random.default_rng(7)
d = rs.beta(1.3, 4, N)
u = rs.random(N)
sig = lambda z: 1 / (1 + np.exp(-z))
p_cheap = 0.99 * sig(-12 * (d - 0.50))
p_strong = 0.99 * sig(-12 * (d - 0.90))
ok_cheap, ok_strong = u < p_cheap, u < p_strong
CHEAP, STRONG = "claude-haiku-5-5", "claude-sonnet-5-5"
TIN, TOUT, ROUTER_IN = 800, 250, 150

def cost(model, tin, tout):
    pi, po = PRICES[model]; return (tin * pi + tout * po) / 1e6

c_cheap, c_strong, c_router = cost(CHEAP, TIN, TOUT), cost(STRONG, TIN, TOUT), cost(CHEAP, ROUTER_IN, 3)
print(f"per query: cheap ${c_cheap:.5f}  strong ${c_strong:.5f}  ({c_strong / c_cheap:.0f}x) | router overhead ${c_router:.6f}")
print(f"all-cheap accuracy {ok_cheap.mean():.3f}   all-strong accuracy {ok_strong.mean():.3f}")
assert ok_strong.mean() > ok_cheap.mean() + 0.05 and (ok_cheap & ~ok_strong).sum() == 0

Now the two policies. Both depend on a **noisy signal**: the router sees `d̂ = d + noise` (a stand-in for a small classifier); the cascade sees the cheap model's *self-reported confidence* `c = p_cheap + noise` (weakly calibrated, as in A08).
We sweep each policy's threshold and plot cost against accuracy.

In [ ]:
def run_router(theta, noise=0.15, seed=11):
    dh = d + np.random.default_rng(seed).normal(0, noise, N)
    to_strong = dh > theta
    acc = np.where(to_strong, ok_strong, ok_cheap).mean()
    cst = (c_router + np.where(to_strong, c_strong, c_cheap)).mean()
    return acc, cst, to_strong.mean()

def run_cascade(tau, noise=0.15, seed=12):
    conf = p_cheap + np.random.default_rng(seed).normal(0, noise, N)
    esc = conf < tau
    acc = np.where(esc, ok_strong, ok_cheap).mean()
    cst = (c_cheap + np.where(esc, c_strong, 0.0)).mean()
    return acc, cst, esc.mean()

base_acc, base_cost = ok_strong.mean(), c_strong
fig, ax = plt.subplots(figsize=(6.5, 3.8))
R = [run_router(th) for th in np.linspace(0.1, 0.9, 17)]; C = [run_cascade(t) for t in np.linspace(0.3, 1.0, 15)]
ax.plot([c * 1000 for _, c, _ in R], [a for a, _, _ in R], "o-", label="router"); ax.plot([c * 1000 for _, c, _ in C], [a for a, _, _ in C], "s-", label="cascade")
ax.scatter([c_cheap * 1000], [ok_cheap.mean()], c="k", marker="v", label="all cheap"); ax.scatter([c_strong * 1000], [base_acc], c="r", marker="^", label="all strong")
ax.set_xlabel("cost per 1,000 queries ($)"); ax.set_ylabel("accuracy"); ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

TOL = 0.03      # business rule: lose at most 3 accuracy points versus all-strong
for name, runs in (("router", R), ("cascade", C)):
    feasible = [(c, a, f) for a, c, f in runs if a >= base_acc - TOL]
    c, a, f = min(feasible)
    print(f"{name:8s} cheapest config within {TOL*100:.0f} points of all-strong: accuracy {a:.3f} (strong {base_acc:.3f}), cost ${c * 1000:.2f}/1k vs ${base_cost * 1000:.2f}/1k "
          f"-> saves {1 - c / base_cost:.0%}, sends {f:.0%} to the strong model")
    assert 1 - c / base_cost > 0.30

**Reading the plot.** The curves are *Pareto frontiers*: pick the cheapest point that still meets your quality floor, and **write the floor down first**. The floor must be set on the metric your users
care about, measured on a labelled sample drawn from *real* traffic: this synthetic workload is only as honest as its difficulty distribution.

The same idea with the library you already know: the router itself is a (cheap) LLM call. Offline, a Python policy plays that classifier; live, the same code asks a small model.

In [ ]:
EASY_HINT = ("reset", "opening hours", "address", "status of")
def router_policy(messages, system, tools):
    q = messages[-1]["content"].lower()
    return say("easy" if any(h in q for h in EASY_HINT) else "hard")

router_llm = get_llm(policy=router_policy)
SYSTEM_ROUTER = "Classify the support query as 'easy' (lookup/FAQ) or 'hard' (needs reasoning over policy). Answer with one word."
demo = ["What is the status of claim 4471?", "My roof leak began before the policy renewed but I only noticed after; is the pre-existing clause relevant?", "How do I reset my portal password?"]
labels = [router_llm.chat([{"role": "user", "content": q}], system=SYSTEM_ROUTER, max_tokens=5).text.strip() for q in demo]
print(list(zip([q[:42] for q in demo], labels)))
if router_llm.mode == "offline":
    assert labels == ["easy", "hard", "easy"]

## 7 · Break it
**7a. A miscalibrated signal.** Both policies are only as good as their signal. Raise the noise on the difficulty/confidence estimate (a distribution shift, a weak classifier) and watch accuracy fall *and* cost rise: a noisy signal escalates the wrong queries.

**7b. The hidden cost of cascades.** Escalated queries pay twice and wait twice: the *tail latency* of a cascade is cheap-latency + strong-latency. If your p95 matters, route (one hop) rather than cascade, or run the cheap check in parallel.

In [ ]:
for noise in (0.05, 0.15, 0.40, 0.80):
    a, c, f = run_cascade(0.8, noise=noise)
    ra, rc, rf = run_router(0.5, noise=noise)
    print(f"signal noise {noise:.2f} | cascade(tau=.8): acc {a:.3f} cost/1k ${c*1000:.2f} esc {f:.0%} | router(theta=.5): acc {ra:.3f} cost/1k ${rc*1000:.2f} strong {rf:.0%}")
a_good, _, _ = run_cascade(0.8, noise=0.05); a_bad, _, _ = run_cascade(0.8, noise=0.80)
assert a_good > a_bad + 0.02, "a noisy confidence signal must hurt cascade quality"

cheap_lat, strong_lat = 1.2, 3.5
print(f"\ncascade latency: easy {cheap_lat:.1f}s, escalated {cheap_lat + strong_lat:.1f}s (illustrative); router: {cheap_lat:.1f}s or {strong_lat:.1f}s + router hop")

## 8 · Putting numbers on Tāmaki's question
Stack the levers on the claims-triage workload (40,000 requests/day: 6,000-token prefix, 300-token note, 250 output tokens) using the cost functions above. The levers act on different parts of the bill, so
they multiply only partly; the code below recomputes cost at each step rather than multiplying headline percentages.

In [ ]:
REQ_PER_DAY = 40000
def daily(model, prefix_mult=1.0, out_tokens=OUT):
    pi, po = PRICES[model]
    per = (PREFIX * prefix_mult + SUFFIX) * pi / 1e6 + out_tokens * po / 1e6
    return per * REQ_PER_DAY

step0 = daily(STRONG)
step1 = daily(STRONG, prefix_mult=READ_MULT * 0.99 + WRITE_MULT * 0.01)          # ~99% cache hits at this volume
esc = run_cascade(0.8)[2]                                                        # share still escalated to the strong model at tau=0.8 (from the sweep above)
print(f"escalation share at tau=0.8: {esc:.0%}")
step2 = esc * daily(STRONG, prefix_mult=READ_MULT * 0.99 + WRITE_MULT * 0.01) + 1.0 * daily(CHEAP, prefix_mult=READ_MULT * 0.99 + WRITE_MULT * 0.01)
for name, v in [("baseline (strong model, no cache)", step0), ("+ prompt caching", step1), ("+ cheap-first cascade", step2)]:
    print(f"{name:36s} ${v:8.2f}/day  ({v / step0:.0%} of baseline)")
assert step1 < 0.5 * step0 and step2 < step1

(Whether the cascade's cheap tier tolerates a cached 6k prefix, and the exact discount, depend on the models' caching rules; this is a planning estimate, and the next step in a real project is to run the candidate configuration through the eval gate from A07 / A10.)

## 9 · Exercises
**Practice (30 min)**
1. Compute the KV-cache bytes per token for a hypothetical model with `L=80, H_kv=8, d_head=128` at fp16 and fp8. How many 32k-token sequences fit in 40 GB of free memory?
2. In `simulate_cache`, change the TTL to 3,600 s (a longer TTL option, if your provider offers one at a different write multiplier, say 2×). At what request rate does the longer TTL start to win?

**Stretch (60 min)**
3. Implement **per-channel** (per-output-column) int4 quantization and compare its output error to per-tensor and group=64 on the same `W`. Which outliers does it not help with?
4. Extend the cascade with a **verifier** that has its own cost and error rate (e.g. a cheap LLM-as-judge that flags wrong answers with 80% recall / 10% false-alarm rate). Does it beat the confidence signal?

**Challenge (2 h, optional LIVE)**
5. Take 100 real or realistic queries from your own domain, label them, run cheap and strong models LIVE, and draw the real Pareto frontier. How far is it from the synthetic one?

In [ ]:
# Starter for exercise 1 — fill in and make the assertions pass
def seqs_that_fit(free_gb, layers, kv_heads, head_dim, seq, bytes_per):
    # TODO: use kv_bytes(...)
    raise NotImplementedError

# n16 = seqs_that_fit(40, 80, 8, 128, 32000, 2); n8 = seqs_that_fit(40, 80, 8, 128, 32000, 1)
# assert n8 >= 2 * n16 - 1 and n16 >= 1
# print(n16, n8)

## ✅ Checkpoint
1. A user complains "the answer starts slowly but then streams fast". Which term is large and name two fixes.
2. Why does adding more concurrent users make *each* user's tokens per second drop even though total throughput rises?
3. Your cache hit rate is 0% although the system prompt is "the same". Give two likely causes.
4. You cut cost 60% with a cascade and the dashboard shows no change in average user rating. Is that enough evidence to ship? What else do you check?

<details><summary>Answers</summary>

1. TTFT (queue + prefill). Fixes: shorten or cache the prefix, reduce load/queueing, route long prompts to a faster tier, stream so the wait is the only wait.
2. The per-step time grows with batch size (more KV and per-sequence work per step), so each sequence gets fewer steps per second; throughput still rises because the shared weight read is amortised.
3. Volatile content early in the prompt (timestamp, user id), reordered tool/schema definitions, different model or parameters, prompt below the minimum cacheable length, or requests spaced further apart than the TTL.
4. No. Averages hide the tail: check per-segment accuracy on a labelled sample (hard queries especially), error types, escalation rate over time (drift), p95 latency, and run it against the golden-set gate.
</details>

## 📌 Key takeaways
- Latency = queue + prefill (TTFT) + decode; **output tokens are the slow, expensive part**, and streaming hides total time but not TTFT.
- Batching trades per-user speed for throughput; **continuous batching** removes idle slots. Offline jobs belong on batch endpoints.
- **KV-cache memory** = `2·L·H_kv·d_head·s·b·p`; it caps concurrency. GQA and KV quantization raise the cap.
- Quantization: 8-bit is nearly free, 4-bit needs grouping or smarter rounding; always re-run your task eval.
- **Prompt caching** pays above roughly one request per TTL and dies on any early-prefix change: put stable text first, volatile text last.
- Routers and cascades save real money when the difficulty signal is good; choose the cheapest config meeting a **pre-declared quality floor**, and re-check it as traffic drifts.

## 📚 Further reading
- Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention* (2023), arXiv:2309.06180
- Frantar et al., *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers* (2022), arXiv:2210.17323
- Lin et al., *AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration* (2023), arXiv:2306.00978
- Ainslie et al., *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints* (2023), arXiv:2305.13245
- Anthropic docs: *Prompt caching*, *Batch processing*, *Pricing*

**Next → A10 · Capstone: Production RAG App with Evals & Guardrails** (combine everything: retrieval, citations, guardrails, an eval gate and a cost report)
